# Selection from WSDB

Following Sergeys advice to use the WSDB to cleanup the data. We will re-run the crossmatch and select the stars that lie within that polygon as K giants

In [1]:
import sqlutilpy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.table import Table

In [30]:
# load credentials
with open('/Users/mncavieres/Documents/2024-1/Delve/credentials_wsdb') as f:
    user, password = f.read().split(', ')

In [36]:
res = sqlutilpy.get('select ra, dec from sdssdr9.phototag limit 10',
                       db='wsdb',host='wsdb.ast.cam.ac.uk', user=user, password=password, asDict=True)

In [2]:
# pip install sqlutilpy astropy
import sqlutilpy
from astropy.table import Table
from collections import Counter


with open('/Users/mncavieres/Documents/2024-1/Delve/credentials_wsdb') as f:
    user, password = f.read().split(', ')

HOST = 'wsdb.ast.cam.ac.uk'
DB   = 'wsdb'


TBL_DELVE = 'delve_dr3.main'   # alias d
TBL_VHS   = 'vhs_1603.des'     # alias v
TBL_GAIA  = 'gaia_source'      # alias g (unqualified)

RADIUS_ARCSEC = 1.0

# giants selection polygon vertices
PX = [1.63, 1.08, 1.54, 2.00, 2.64, 2.04]
PY = [2.43, 2.17, 2.61, 2.95, 3.28, 2.65]

# helps
def _parse_schema_table(qualified):
    if '.' in qualified:
        s, t = qualified.split('.', 1)
        return s, t
    return None, qualified

def _resolve_schema_for(table_name):
    q = """
    SELECT table_schema, COUNT(*) AS ncols
    FROM information_schema.columns
    WHERE table_name = %s
    GROUP BY table_schema
    ORDER BY ncols DESC
    LIMIT 1
    """
    res = sqlutilpy.get(q, db=DB, host=HOST, user=user, password=password,
                        asDict=True, params=(table_name,))
    if res and 'table_schema' in res and len(res['table_schema']) > 0:
        return res['table_schema'][0]
    return None

def _list_columns(schema, table):
    if schema is None:
        q = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = %s
        ORDER BY ordinal_position
        """
        r = sqlutilpy.get(q, db=DB, host=HOST, user=user, password=password,
                          asDict=True, params=(table,))
    else:
        q = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position
        """
        r = sqlutilpy.get(q, db=DB, host=HOST, user=user, password=password,
                          asDict=True, params=(schema, table))
    return list(r['column_name'])

# parse / resolve schemas
d_schema, d_table = _parse_schema_table(TBL_DELVE)
v_schema, v_table = _parse_schema_table(TBL_VHS)
g_schema, g_table = _parse_schema_table(TBL_GAIA)
if g_schema is None:
    g_schema = _resolve_schema_for(g_table)

# get column lists
cols_d = _list_columns(d_schema, d_table)
cols_v = _list_columns(v_schema, v_table)
cols_g = _list_columns(g_schema, g_table)

# figure out which names are repeated across surveys
all_names = cols_d + cols_v + cols_g
name_counts = Counter(all_names)

SUFFIX = {'d': 'delve', 'v': 'vhs', 'g': 'gaia'}

def _qualified_ref(alias, col):
    return f'{alias}."{col}"'

def _alias_name(col, alias_letter):
    # keep original name if unique; add suffix only if repeated
    return col if name_counts[col] == 1 else f"{col}_{SUFFIX[alias_letter]}"

def _build_select_list(alias_letter, cols):
    return ",\n    ".join(
        f'{_qualified_ref(alias_letter, c)} AS "{_alias_name(c, alias_letter)}"'
        for c in cols
    )

sel_d = _build_select_list('d', cols_d)
sel_v = _build_select_list('v', cols_v)
sel_g = _build_select_list('g', cols_g)

tbl_d = f'{d_schema}.{d_table}'
tbl_v = f'{v_schema}.{v_table}'
tbl_g = f'{g_schema+"." if g_schema else ""}{g_table}'

# build SQL query, updated some columns to use DELVE DR3 instead of DR2
SQL = f"""
WITH params AS (
  SELECT {RADIUS_ARCSEC}/3600.0 AS rdeg
),
poly AS (
  SELECT
    ARRAY[{", ".join(str(x) for x in PX)}]::double precision[] AS px,
    ARRAY[{", ".join(str(y) for y in PY)}]::double precision[] AS py
),
x AS (
  SELECT
    {sel_d},
    {sel_v},
    {sel_g},
    q3c_dist(d.ra, d.dec, v.ra, v.dec) AS dist_d_v_deg,
    q3c_dist(d.ra, d.dec, g.ra, g.dec) AS dist_d_g_deg,
    (d.wavg_mag_psf_g - d.wavg_mag_psf_i) AS g_i,
    (d.wavg_mag_psf_i - v.kapercormag3)   AS i_K
  FROM {tbl_d} AS d
  JOIN LATERAL (
    SELECT v.*
    FROM {tbl_v} AS v, params p
    WHERE q3c_join(d.ra, d.dec, v.ra, v.dec, p.rdeg)
      AND v.mergedclass = -1
      AND v.jerrorbit = 0
      AND v.kerrorbit = 0
      AND v.prim = 1
      AND v.javconf > 95
      AND v.kavconf > 95
      AND (v.jx < 8800 OR v.jy < 12300)
      AND v.japercormag4 BETWEEN 12 AND 18
      AND v.kapercormag4 BETWEEN 11 AND 19
    ORDER BY q3c_dist(d.ra, d.dec, v.ra, v.dec)
    LIMIT 1
  ) v ON TRUE
  JOIN LATERAL (
    SELECT g.*
    FROM {tbl_g} AS g, params p
    WHERE q3c_join(d.ra, d.dec, g.ra, g.dec, p.rdeg)
      AND g.parallax < 0.1
    ORDER BY q3c_dist(d.ra, d.dec, g.ra, g.dec)
    LIMIT 1
  ) g ON TRUE
  WHERE
    d.ext_coadd = 1
    AND (d.wavg_mag_psf_g - d.wavg_mag_psf_i) > 1.2
    AND (d.wavg_mag_psf_g - d.wavg_mag_psf_i) < 2.6
    AND d.s_extractor_flags_g = 0
    AND d.s_extractor_flags_i = 0
    AND d.s_extractor_flags_r = 0
    AND d.s_extractor_flags_z = 0
)
SELECT *
FROM x
WHERE (
  SELECT (SUM(
           CASE
             WHEN ((p.py[i] > x.i_K) <> (p.py[j] > x.i_K))
              AND (x.g_i < ((p.px[j]-p.px[i]) * (x.i_K - p.py[i])
                            / NULLIF((p.py[j]-p.py[i]), 0) + p.px[i]))
             THEN 1 ELSE 0 END
         ) % 2)
  FROM poly AS p,
       generate_subscripts(p.px,1) AS i,
       LATERAL (SELECT CASE WHEN i < array_length(p.px,1) THEN i+1 ELSE 1 END AS j) AS nxt
) = 1;
"""


res = sqlutilpy.get(SQL, db=DB, host=HOST, user=user, password=password, asDict=True)
tab = Table(res)
tab.write('delve_vhs_gaia_polygon.fits', overwrite=True)
print(f"Wrote {len(tab)} rows × {len(tab.colnames)} cols to delve_vhs_gaia_polygon.fits")


KeyboardInterrupt: 

In [6]:
# pip install sqlutilpy astropy
import sqlutilpy
from astropy.table import Table
from collections import Counter

# --- credentials ---
with open('/Users/mncavieres/Documents/2024-1/Delve/credentials_wsdb') as f:
    user, password = f.read().split(', ')

HOST = 'wsdb.ast.cam.ac.uk'
DB   = 'wsdb'

# --- tables ---
TBL_DELVE = 'delve_dr3.main'   # alias d
TBL_VHS   = 'vhs_1603.des'     # alias v
TBL_GAIA  = 'gaia_source'      # alias g

# --- match + polygon ---
RADIUS_ARCSEC = 1.0
PX = [1.63, 1.08, 1.54, 2.00, 2.64, 2.04]  # polygon x=g_i
PY = [2.43, 2.17, 2.61, 2.95, 3.28, 2.65]  # polygon y=i_K

# ---------- helpers to alias ONLY duplicate column names ----------
def _parse_schema_table(qualified):
    return qualified.split('.', 1) if '.' in qualified else (None, qualified)

def _resolve_schema_for(table_name):
    q = """
    SELECT table_schema, COUNT(*) AS ncols
    FROM information_schema.columns
    WHERE table_name = %s
    GROUP BY table_schema
    ORDER BY ncols DESC
    LIMIT 1
    """
    r = sqlutilpy.get(q, db=DB, host=HOST, user=user, password=password, asDict=True, params=(table_name,))
    return r['table_schema'][0] if r and 'table_schema' in r and len(r['table_schema']) else None

def _list_columns(schema, table):
    if schema:
        q = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position
        """
        r = sqlutilpy.get(q, db=DB, host=HOST, user=user, password=password, asDict=True, params=(schema, table))
    else:
        q = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = %s
        ORDER BY ordinal_position
        """
        r = sqlutilpy.get(q, db=DB, host=HOST, user=user, password=password, asDict=True, params=(table,))
    return list(r['column_name'])

d_schema, d_table = _parse_schema_table(TBL_DELVE)
v_schema, v_table = _parse_schema_table(TBL_VHS)
g_schema, g_table = _parse_schema_table(TBL_GAIA)
if g_schema is None:
    g_schema = _resolve_schema_for(g_table)

cols_d = _list_columns(d_schema, d_table)
cols_v = _list_columns(v_schema, v_table)
cols_g = _list_columns(g_schema, g_table)

name_counts = Counter(cols_d + cols_v + cols_g)
SUFFIX = {'d':'delve','v':'vhs','g':'gaia'}

def _alias_name(col, alias_letter):
    return col if name_counts[col] == 1 else f"{col}_{SUFFIX[alias_letter]}"

def _sel(alias_sql, alias_letter, cols):
    return ",\n    ".join(f'{alias_sql}."{c}" AS "{_alias_name(c, alias_letter)}"' for c in cols)

sel_d     = _sel('d',     'd', cols_d)
sel_vfull = _sel('vfull', 'v', cols_v)
sel_gfull = _sel('gfull', 'g', cols_g)

tbl_d = f'{d_schema}.{d_table}'
tbl_v = f'{v_schema}.{v_table}'
tbl_g = f'{g_schema+"." if g_schema else ""}{g_table}'

# ---------- single optimized SQL following WSDB/Q3C guidance ----------
SQL = f"""
WITH params AS (
  SELECT {RADIUS_ARCSEC}/3600.0 AS rdeg
),
-- (Optional) add a sky prefilter on DELVE using q3c_radial_query/q3c_poly_query/box if needed
--   AND q3c_radial_query(d.ra, d.dec, <ra0>, <dec0>, <rdeg>)
--   AND q3c_poly_query  (d.ra, d.dec, ARRAY[...])
--   AND d.ra BETWEEN ... AND ... AND d.dec BETWEEN ... AND ...
d AS MATERIALIZED (
  SELECT *
  FROM {tbl_d} d
  WHERE     
    d.ext_coadd = 1
    AND d.s_extractor_flags_g = 0
    AND d.s_extractor_flags_i = 0
    AND d.s_extractor_flags_r = 0
    AND d.s_extractor_flags_z = 0
    AND (d.wavg_mag_psf_g - d.wavg_mag_psf_i) > 1.2
    AND (d.wavg_mag_psf_g - d.wavg_mag_psf_i) < 2.6

),
-- Per WSDB docs: filter the second tables in separate MATERIALIZED CTEs, then spatially join
vhs0 AS MATERIALIZED (
  SELECT ctid AS v_ctid, ra, dec, kapercormag3, japercormag4, kapercormag4
  FROM {tbl_v} v
  WHERE v.mergedclass = -1
    AND v.jerrorbit = 0
    AND v.kerrorbit = 0
    AND v.prim = 1
    AND v.javconf > 95
    AND v.kavconf > 95
    AND (v.jx < 8800 OR v.jy < 12300)
    AND v.japercormag4 BETWEEN 12 AND 18
    AND v.kapercormag4 BETWEEN 11 AND 19
),
gaia0 AS MATERIALIZED (
  SELECT source_id, ra, dec, parallax
  FROM {tbl_g} g
  WHERE g.parallax < 0.1
),
-- Color–color polygon in (g_i, i_K) space
poly AS (
  SELECT
    ARRAY[{", ".join(str(x) for x in PX)}]::double precision[] AS px,
    ARRAY[{", ".join(str(y) for y in PY)}]::double precision[] AS py
)
SELECT
  {sel_d},
  {sel_vfull},
  {sel_gfull},
  -- diagnostics + derived colors
  q3c_dist(d.ra, d.dec, vbest.ra, vbest.dec) AS dist_d_v_deg,
  q3c_dist(d.ra, d.dec, gbest.ra, gbest.dec) AS dist_d_g_deg,
  (d.wavg_mag_psf_g - d.wavg_mag_psf_i) AS g_i,
  (d.wavg_mag_psf_i - vbest.kapercormag3) AS i_K
FROM d
-- Nearest VHS using q3c_join (WSDB “Nearest neighbor” pattern)
JOIN LATERAL (
  SELECT v0.v_ctid, v0.ra, v0.dec, v0.kapercormag3
  FROM vhs0 v0, params p
  WHERE q3c_join(d.ra, d.dec, v0.ra, v0.dec, p.rdeg)
  ORDER BY q3c_dist(d.ra, d.dec, v0.ra, v0.dec)
  LIMIT 1
) vbest ON TRUE
-- Join back to full VHS to pull all columns (use a stable PK if available; CTID is OK within one query)
JOIN {tbl_v} vfull ON vfull.ctid = vbest.v_ctid
-- Nearest Gaia using q3c_join
JOIN LATERAL (
  SELECT g0.source_id, g0.ra, g0.dec
  FROM gaia0 g0, params p
  WHERE q3c_join(d.ra, d.dec, g0.ra, g0.dec, p.rdeg)
  ORDER BY q3c_dist(d.ra, d.dec, g0.ra, g0.dec)
  LIMIT 1
) gbest ON TRUE
JOIN {tbl_g} gfull ON gfull.source_id = gbest.source_id
-- Polygon inclusion (ray-casting) applied AFTER spatial joins, as a final filter
WHERE (
  SELECT (SUM(
           CASE
             WHEN ((p.py[i] > (d.wavg_mag_psf_i - vbest.kapercormag3))
                   <> (p.py[j] > (d.wavg_mag_psf_i - vbest.kapercormag3)))
              AND ((d.wavg_mag_psf_g - d.wavg_mag_psf_i) <
                  ((p.px[j]-p.px[i]) * ((d.wavg_mag_psf_i - vbest.kapercormag3) - p.py[i])
                   / NULLIF((p.py[j]-p.py[i]), 0) + p.px[i]))
             THEN 1 ELSE 0 END
         ) % 2)
  FROM poly AS p,
       generate_subscripts(p.px,1) AS i,
       LATERAL (SELECT CASE WHEN i < array_length(p.px,1) THEN i+1 ELSE 1 END AS j) AS nxt
) = 1;
"""

# ---- run once & save ----
res = sqlutilpy.get(SQL, db=DB, host=HOST, user=user, password=password, asDict=True)
tab = Table(res)
tab.write('delve_vhs_gaia_singlequery_wsdb.fits', overwrite=True)
print(f"Wrote {len(tab)} rows × {len(tab.colnames)} cols -> delve_vhs_gaia_singlequery_wsdb.fits")


KeyboardInterrupt: 